
# Collecte OSM locale (extrait Geofabrik) — sans appels réseau

Ce notebook remplace la version Overpass API par une lecture **locale** de
l'extrait Algérie téléchargé sur Geofabrik (`algeria-latest.osm.pbf`).
Plus de rate-limit, plus de timeout, plus de dépendance à la disponibilité
d'un serveur public — tout est filtré en mémoire sur ta machine.

**Avant de lancer :**
1. Télécharge `algeria-latest.osm.pbf` depuis
   https://download.geofabrik.de/africa/algeria.html
2. `pip install osmium --break-system-packages`
3. Renseigne `PBF_PATH` ci-dessous vers le fichier téléchargé.

**Changement important par rapport à la version Overpass par wilaya :**
On ne peut plus garantir `wilaya_name` pour chaque élément comme avant
(là où on interrogeait wilaya par wilaya, on connaissait la wilaya à coup
sûr). Ici, tout le pays est lu en un seul passage, donc la wilaya est
retrouvée via le tag `addr:city`/`is_in` de l'élément (quand il existe) et
un matching flou **national** contre la table `communes`. C'est moins fiable
qu'avant, mais reste largement mieux que la situation initiale sur
`cabinet médical`/`laboratoire d'analyses` (qui n'avaient aucune wilaya).

Si tu veux plus tard une précision garantie, l'étape suivante serait
d'extraire les polygones de wilaya (relations `admin_level=4`) du même
fichier `.pbf` et de faire un vrai test point-dans-polygone (avec
`shapely`) — plus lourd à mettre en place, on peut le faire dans un second
temps si le matching texte s'avère insuffisant.


In [2]:
import sys
print(sys.executable)

c:\Users\HP\anaconda3\python.exe


In [2]:

import sqlite3
import unicodedata
import difflib
import osmium

DB_PATH = "../data/dasec_prospection.db"
PBF_PATH = "../data/algeria-latest.osm.pbf"   # <-- adapte le chemin si besoin

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print("Connecté à", DB_PATH)


Connecté à ../data/dasec_prospection.db


## 1. Communes de référence (pour le matching wilaya/commune)

In [5]:

def normaliser(texte: str) -> str:
    if not texte:
        return ""
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return texte.lower().strip()

# Index national : nom de commune normalisé -> (commune_id, commune_name, wilaya_name)
cursor = conn.execute("SELECT id, commune_name, wilaya_name FROM communes WHERE commune_name IS NOT NULL")
INDEX_COMMUNES = {}
for row in cursor.fetchall():
    INDEX_COMMUNES[normaliser(row["commune_name"])] = (row["id"], row["commune_name"], row["wilaya_name"])

print(f"{len(INDEX_COMMUNES)} communes indexées.")

def matcher_commune_nationale(texte_brut: str):
    """Matching flou national (pas de contexte wilaya préalable ici,
    contrairement à la version Overpass par wilaya). Retourne
    (commune_id, commune_name, wilaya_name) ou (None, None, None)."""
    if not texte_brut:
        return None, None, None
    cible = normaliser(texte_brut)
    if cible in INDEX_COMMUNES:
        return INDEX_COMMUNES[cible]
    proches = difflib.get_close_matches(cible, INDEX_COMMUNES.keys(), n=1, cutoff=0.8)
    if proches:
        return INDEX_COMMUNES[proches[0]]
    return None, None, None


1500 communes indexées.


## 2. Config des groupes à collecter (matchers de tags, équivalents aux filtres Overpass précédents)

In [6]:

def classifier_hopital(nom: str) -> str:
    n = normaliser(nom)
    if "chu" in n:
        return "CHU"
    if "ehs" in n:
        return "EHS"
    if "eph" in n or "etablissement public hospitalier" in n:
        return "EPH"
    return "hôpital"

# Chaque groupe : secteur, sous_secteur (None si classifié après coup via
# `classifier`), et `match(tags: dict) -> bool`.
SOUS_SECTEURS_A_COLLECTER = [
    {
        "secteur": "santé", "sous_secteur": None, "classifier": classifier_hopital,
        "match": lambda t: t.get("amenity") == "hospital",
    },
    {
        "secteur": "santé", "sous_secteur": "clinique privée",
        "match": lambda t: t.get("healthcare") == "clinic" or t.get("amenity") == "clinic",
    },
    {
        "secteur": "santé", "sous_secteur": "laboratoire d'analyses",
        "match": lambda t: t.get("healthcare") == "laboratory",
    },
    {
        "secteur": "santé", "sous_secteur": "centre de transfusion sanguine",
        "match": lambda t: t.get("healthcare") == "blood_donation" or "transfusion" in normaliser(t.get("name", "")),
    },
    {
        "secteur": "santé", "sous_secteur": "opticien",
        "match": lambda t: t.get("shop") == "optician",
    },
    {
        "secteur": "santé", "sous_secteur": "cabinet dentaire",
        "match": lambda t: t.get("amenity") == "dentist" or t.get("healthcare") == "dentist",
    },
    {
        "secteur": "santé", "sous_secteur": "cabinet médical",
        "match": lambda t: t.get("amenity") == "doctors" or t.get("healthcare") == "doctor",
    },
    {
        "secteur": "assurance", "sous_secteur": "assurance privée",
        "match": lambda t: t.get("office") == "insurance",
    },
    {
        "secteur": "assurance", "sous_secteur": "CNAS",
        "match": lambda t: "cnas" in normaliser(t.get("name", "")),
    },
    {
        "secteur": "assurance", "sous_secteur": "CASNOS",
        "match": lambda t: "casnos" in normaliser(t.get("name", "")),
    },
    {
        "secteur": "juridique", "sous_secteur": "cabinet d'avocat",
        "match": lambda t: t.get("office") == "lawyer",
    },
    {
        "secteur": "juridique", "sous_secteur": "cabinet comptable",
        "match": lambda t: t.get("office") == "accountant",
    },
    {
        "secteur": "juridique", "sous_secteur": "notaire",
        "match": lambda t: t.get("office") == "notary",
    },
    {
        # Bruyant par nature -> à trier après import plutôt qu'à la collecte.
        "secteur": "industrie", "sous_secteur": "entreprise industrielle",
        "match": lambda t: t.get("building") == "industrial" and bool(t.get("name")),
    },
    {
        "secteur": "étatique", "sous_secteur": "mairie",
        "match": lambda t: t.get("amenity") == "townhall",
    },
    {
        "secteur": "étatique", "sous_secteur": "siège de wilaya",
        "match": lambda t: "wilaya de" in normaliser(t.get("name", "")),
    },
]

def trouver_groupe(tags: dict):
    for groupe in SOUS_SECTEURS_A_COLLECTER:
        if groupe["match"](tags):
            return groupe
    return None

print(f"{len(SOUS_SECTEURS_A_COLLECTER)} groupes configurés.")


16 groupes configurés.


## 3. Lecture du fichier .pbf (un seul passage, nœuds + chemins)

In [7]:

class CollecteHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.resultats = []  # liste de dicts prêts à insérer

    def _extraire_commun(self, tags: dict, source_id: str, lat, lon, groupe) -> dict:
        return {
            "source_id": source_id,
            "secteur": groupe["secteur"],
            "sous_secteur": groupe["classifier"](tags.get("name")) if groupe.get("classifier") else groupe["sous_secteur"],
            "nom": tags.get("name"),
            "telephone": tags.get("contact:phone") or tags.get("phone"),
            "email": tags.get("contact:email") or tags.get("email"),
            "site_web": tags.get("contact:website") or tags.get("website"),
            "facebook": tags.get("contact:facebook"),
            "latitude": lat,
            "longitude": lon,
            "commune_brute": tags.get("addr:city") or tags.get("addr:municipality") or tags.get("is_in"),
            "adresse": tags.get("addr:full") or tags.get("addr:street"),
        }

    def node(self, n):
        tags = {tag.k: tag.v for tag in n.tags}
        if not tags:
            return
        groupe = trouver_groupe(tags)
        if groupe is None:
            return
        if not n.location.valid():
            return
        self.resultats.append(self._extraire_commun(tags, f"node/{n.id}", n.location.lat, n.location.lon, groupe))

    def way(self, w):
        tags = {tag.k: tag.v for tag in w.tags}
        if not tags:
            return
        groupe = trouver_groupe(tags)
        if groupe is None:
            return
        lats, lons = [], []
        for node_ref in w.nodes:
            if node_ref.location.valid():
                lats.append(node_ref.location.lat)
                lons.append(node_ref.location.lon)
        if not lats:
            return
        lat = sum(lats) / len(lats)
        lon = sum(lons) / len(lons)
        self.resultats.append(self._extraire_commun(tags, f"way/{w.id}", lat, lon, groupe))


print("Lecture du fichier .pbf en cours (peut prendre quelques minutes selon la taille)...")

idx = osmium.index.create_map("sparse_mem_array")
lh = osmium.NodeLocationsForWays(idx)
lh.ignore_errors()  # ignore les ways dont un nœud référencé serait absent de l'extrait

handler = CollecteHandler()
osmium.apply(osmium.io.Reader(PBF_PATH), lh, handler)

print(f"{len(handler.resultats)} éléments correspondant à un groupe trouvés.")


Lecture du fichier .pbf en cours (peut prendre quelques minutes selon la taille)...
8028 éléments correspondant à un groupe trouvés.


In [4]:

cur = conn.cursor()
# Dry run d'abord : voir ce qui serait reclassé
cur.execute("""
    SELECT id, nom FROM entreprises
    WHERE secteur = 'juridique' AND sous_secteur = "cabinet d'avocat"
      AND (nom LIKE '%notaire%' OR nom LIKE '%Notaire%' OR nom LIKE '%NOTAIRE%' OR nom LIKE '%موثق%')
""")
candidats = cur.fetchall()

print(f"{len(candidats)} lignes seraient reclassées cabinet d'avocat -> notaire :")
for id_, nom in candidats:
    print(f"  id={id_}  {nom}")

conn.close()

12 lignes seraient reclassées cabinet d'avocat -> notaire :
  id=5475  Notaire
  id=5636  Notaire
  id=6988  Notaire
  id=6991  Notaire
  id=7002  Notaire
  id=7112  موثق
  id=7114  موثق-فار
  id=7191  notaire/ avocat
  id=7420  الموثق حسن محمد إقبال
  id=7449  موثق
  id=7463  موثق
  id=9817  Notaire Rahmania Abdelkader


In [5]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
    UPDATE entreprises
    SET sous_secteur = 'notaire'
    WHERE secteur = 'juridique' AND sous_secteur = "cabinet d'avocat"
      AND (nom LIKE '%notaire%' OR nom LIKE '%Notaire%' OR nom LIKE '%NOTAIRE%' OR nom LIKE '%موثق%')
""")
conn.commit()
print(f"{cur.rowcount} lignes reclassées.")
conn.close()

12 lignes reclassées.


In [7]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM entreprises WHERE secteur='juridique' AND sous_secteur='notaire'")
print(cur.fetchone()[0])
conn.close()

29


## 4. Insertion dédupliquée en base

In [8]:

def deja_importe(source_id: str) -> bool:
    row = conn.execute("SELECT id FROM entreprises WHERE source_id = ?", (source_id,)).fetchone()
    return row is not None

def inserer_etablissement(champs: dict) -> bool:
    if deja_importe(champs["source_id"]):
        return False

    nom = champs["nom"] or f"{champs['sous_secteur']} (sans nom)"
    commune_id, commune_name, wilaya_name = matcher_commune_nationale(champs["commune_brute"])

    valeurs = (
        nom, champs["secteur"], champs["sous_secteur"], champs["telephone"], champs["email"],
        champs["site_web"], champs["facebook"], champs["latitude"], champs["longitude"],
        commune_id, commune_name or champs["commune_brute"], wilaya_name,
        champs["adresse"], champs["source_id"],
    )

    try:
        conn.execute("""
            INSERT INTO entreprises (
                nom, secteur, sous_secteur, telephone, email, site_web, facebook,
                latitude, longitude, commune_id, commune_brute, wilaya_name,
                adresse, source, source_id
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'osm', ?)
        """, valeurs)
        return True
    except sqlite3.IntegrityError:
        source_unique = f"osm#{champs['source_id']}"
        try:
            conn.execute("""
                INSERT INTO entreprises (
                    nom, secteur, sous_secteur, telephone, email, site_web, facebook,
                    latitude, longitude, commune_id, commune_brute, wilaya_name,
                    adresse, source, source_id
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, valeurs[:-1] + (source_unique, champs["source_id"]))
            return True
        except sqlite3.IntegrityError as e:
            print(f"  [ignoré définitivement] {champs['source_id']} : {e}")
            return False


inseres = 0
sans_wilaya = 0

for champs in handler.resultats:
    if inserer_etablissement(champs):
        inseres += 1
        if not champs.get("commune_brute"):
            sans_wilaya += 1

conn.commit()
print(f"{inseres} nouveaux établissements insérés.")
print(f"Dont {sans_wilaya} sans tag commune exploitable (wilaya_name restera NULL pour ceux-là).")


7576 nouveaux établissements insérés.
Dont 5470 sans tag commune exploitable (wilaya_name restera NULL pour ceux-là).


## 5. Vérification post-import

In [9]:

cursor = conn.execute("""
    SELECT secteur, sous_secteur,
           COUNT(DISTINCT wilaya_name) as nb_wilayas,
           SUM(CASE WHEN wilaya_name IS NULL THEN 1 ELSE 0 END) as sans_wilaya,
           COUNT(*) as total
    FROM entreprises
    WHERE source LIKE 'osm%'
    GROUP BY secteur, sous_secteur
    ORDER BY nb_wilayas ASC
""")
for row in cursor.fetchall():
    print(f"{row['secteur']:12} {row['sous_secteur'] or '—':30} {row['nb_wilayas']:3} wilayas   "
          f"{row['sans_wilaya']:4} sans wilaya   {row['total']:5} total")

conn.close()


santé        centre de transfusion sanguine   1 wilayas      7 sans wilaya       8 total
santé        assurance privée                 2 wilayas      0 sans wilaya      21 total
juridique    cabinet comptable                3 wilayas     14 sans wilaya      18 total
santé        EHS                              3 wilayas      3 sans wilaya       7 total
assurance    CASNOS                           4 wilayas     23 sans wilaya      27 total
juridique    notaire                          4 wilayas     12 sans wilaya      17 total
santé        CHU                              5 wilayas      2 sans wilaya       7 total
santé        EPH                             10 wilayas     12 sans wilaya      26 total
assurance    CNAS                            11 wilayas     72 sans wilaya      86 total
juridique    cabinet d'avocat                11 wilayas    153 sans wilaya     169 total
industrie    entreprise industrielle         18 wilayas    211 sans wilaya     272 total
étatique     siège de

In [11]:
import sqlite3

conn = sqlite3.connect("../data/dasec_prospection.db")
conn.row_factory = sqlite3.Row

# Total général
total = conn.execute("SELECT COUNT(*) as n FROM entreprises").fetchone()["n"]
print(f"Total entreprises en base : {total}\n")

# Répartition par secteur puis sous-secteur
cursor = conn.execute("""
    SELECT secteur, sous_secteur, COUNT(*) as n
    FROM entreprises
    GROUP BY secteur, sous_secteur
    ORDER BY secteur, n DESC
""")

secteur_actuel = None
for row in cursor.fetchall():
    if row["secteur"] != secteur_actuel:
        secteur_actuel = row["secteur"]
        print(f"\n--- {secteur_actuel or '(secteur non renseigné)'} ---")
    print(f"  {row['sous_secteur'] or '(sous-secteur non renseigné)':35} {row['n']:6}")

conn.close()

Total entreprises en base : 12369


--- assurance ---
  assurance privée                       462
  CNAS                                    86
  CASNOS                                  27

--- industrie ---
  entreprise industrielle                272

--- juridique ---
  cabinet d'avocat                       169
  cabinet comptable                       18
  notaire                                 17

--- santé ---
  cabinet médical                       3772
  clinique privée                       1807
  pharmacie                             1756
  hôpital                               1451
  cabinet dentaire                       737
  opticien                               218
  laboratoire d'analyses                 166
  EPH                                    105
  EHS                                     37
  CHU                                     26
  assurance privée                        21
  centre de transfusion sanguine           8
  CAC                                 

In [13]:
import sqlite3

conn = sqlite3.connect("../data/dasec_prospection.db")
conn.row_factory = sqlite3.Row 
cursor = conn.execute("""
    SELECT nom, wilaya_name, source FROM entreprises
    WHERE secteur = 'santé' AND sous_secteur = 'assurance privée'
""")
for row in cursor.fetchall():
    print(dict(row))

{'nom': "CNAS d'Ouzellaguen", 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'SAA', 'wilaya_name': 'Tizi Ouzou', 'source': 'osm'}
{'nom': 'Assurance Agriculture', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'SAA', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'CNAS Béjaïa', 'wilaya_name': 'Béjaïa', 'source': 'osm#480'}
{'nom': "agence d'assurance GAM Assurance", 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': '2a Bejaia 0616 AGD ROUHA', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'SAA', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'Caisse Régionale de Mutualité Agricole', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'SAA', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'Algérienne des assurances', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': "Compagnie Algérienne d'assurance et de réassurance", 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'CAAT / TALA Assurance', 'wilaya_name': 'Béjaïa', 'source': 'osm'}
{'nom': 'MAATEC Assurance', 'wilaya_na

In [14]:
conn.execute("""
    UPDATE entreprises SET secteur = 'assurance'
    WHERE secteur = 'santé' AND sous_secteur = 'assurance privée'
""")
conn.commit()

In [16]:
cursor = conn.execute("""
    SELECT id, nom, source, source_id, latitude, longitude
    FROM entreprises
    WHERE sous_secteur = 'assurance privée'
      AND commune_brute IN ('Oum El Bouaghi', 'Batna', 'Béjaïa', 'Amizour')
    ORDER BY nom, commune_brute
""")
for row in cursor.fetchall():
    print(dict(row))

{'id': 8362, 'nom': '2a', 'source': 'osm', 'source_id': 'node/5588992570', 'latitude': 35.872389, 'longitude': 7.123269}
{'id': 482, 'nom': '2a Bejaia 0616 AGD ROUHA', 'source': 'osm', 'source_id': '5096262023', 'latitude': 36.7478898, 'longitude': 5.0661617}
{'id': 486, 'nom': 'Algérienne des assurances', 'source': 'osm', 'source_id': '5702691637', 'latitude': 36.7515068, 'longitude': 5.0644789}
{'id': 7083, 'nom': 'CAAR', 'source': 'osm', 'source_id': 'node/4585438111', 'latitude': 35.5515732, 'longitude': 6.1723908}
{'id': 8267, 'nom': 'CAAT', 'source': 'osm', 'source_id': 'node/5518496958', 'latitude': 35.8748151, 'longitude': 7.1197991}
{'id': 12279, 'nom': 'CAAT', 'source': 'osm#way/1009382009', 'source_id': 'way/1009382009', 'latitude': 35.87475204, 'longitude': 7.119726559999999}
{'id': 8361, 'nom': 'CIAR', 'source': 'osm', 'source_id': 'node/5588992569', 'latitude': 35.8728588, 'longitude': 7.1233993}
{'id': 10663, 'nom': 'CNAS', 'source': 'osm', 'source_id': 'way/294267641', 

In [19]:
import sqlite3
import unicodedata
import difflib

conn = sqlite3.connect("../data/dasec_prospection.db")
conn.row_factory = sqlite3.Row

def normaliser(texte):
    if not texte:
        return ""
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return texte.lower().strip()

cursor = conn.execute("""
    SELECT id, nom, secteur, sous_secteur, source_id,
           ROUND(latitude, 4) as lat_r, ROUND(longitude, 4) as lon_r
    FROM entreprises
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""")

par_position = {}
for row in cursor.fetchall():
    cle = (row["lat_r"], row["lon_r"], row["secteur"])
    par_position.setdefault(cle, []).append(dict(row))

vrais_doublons = []
for cle, groupe in par_position.items():
    if len(groupe) < 2:
        continue
    for i in range(len(groupe)):
        for j in range(i + 1, len(groupe)):
            n1, n2 = normaliser(groupe[i]["nom"]), normaliser(groupe[j]["nom"])
            similarite = difflib.SequenceMatcher(None, n1, n2).ratio()
            if similarite >= 0.85:  # noms quasi identiques uniquement
                vrais_doublons.append((groupe[i], groupe[j], round(similarite, 2)))

print(f"{len(vrais_doublons)} vrais doublons probables (nom similaire à 85%+ ET même position).")
for a, b, sim in vrais_doublons[:40]:
    print(f"  [{sim}] {a['id']} ({a['nom']}) <-> {b['id']} ({b['nom']})")

2433 vrais doublons probables (nom similaire à 85%+ ET même position).
  [1.0] 113 (Hôpital Arassa) <-> 5011 (Hôpital Arassa)
  [1.0] 114 (polyclinique Souk El Tenine) <-> 4803 (polyclinique Souk El Tenine)
  [1.0] 115 (Hôpital orthopédique) <-> 4804 (Hôpital orthopédique)
  [1.0] 116 (عيادة متعددة الخدمات أيث شبانة) <-> 5022 (عيادة متعددة الخدمات أيث شبانة)
  [1.0] 117 (Oued achalal) <-> 4892 (Oued achalal)
  [1.0] 118 (Hôpital Souk El Tenine مستشفى سوق الاثنين) <-> 4805 (Hôpital Souk El Tenine مستشفى سوق الاثنين)
  [1.0] 119 (TAKLINIKT LEFLAY) <-> 4806 (TAKLINIKT LEFLAY)
  [1.0] 172 (Docteur Dentiste Stambouli) <-> 5923 (Docteur Dentiste Stambouli)
  [1.0] 173 (Dr. Oulebsir) <-> 6043 (Dr. Oulebsir)
  [1.0] 174 (Dentiste les 4 chemins) <-> 6056 (Dentiste les 4 chemins)
  [1.0] 175 (Dentiste Mouloua) <-> 6897 (Dentiste Mouloua)
  [1.0] 176 (Dentiste Boumzoud) <-> 6935 (Dentiste Boumzoud)
  [1.0] 177 (Dentiste Ourabah) <-> 7253 (Dentiste Ourabah)
  [1.0] 178 (Cabinet Docteur Chamekh) <-

In [21]:
import sqlite3

conn = sqlite3.connect("../data/dasec_prospection.db")
conn.row_factory = sqlite3.Row

def score_completude(ligne: dict) -> int:
    """Compte le nombre de champs utiles remplis, pour choisir laquelle garder."""
    champs = ["telephone", "email", "site_web", "facebook", "adresse", "wilaya_name", "commune_id"]
    return sum(1 for c in champs if ligne.get(c))

# Reconstruire les grappes (union des paires qui partagent un id)
grappes = []
index_id_vers_grappe = {}

for a, b, sim in vrais_doublons:
    ga = index_id_vers_grappe.get(a["id"])
    gb = index_id_vers_grappe.get(b["id"])
    if ga is None and gb is None:
        nouvelle = {a["id"], b["id"]}
        grappes.append(nouvelle)
        index_id_vers_grappe[a["id"]] = nouvelle
        index_id_vers_grappe[b["id"]] = nouvelle
    elif ga is not None and gb is None:
        ga.add(b["id"])
        index_id_vers_grappe[b["id"]] = ga
    elif gb is not None and ga is None:
        gb.add(a["id"])
        index_id_vers_grappe[a["id"]] = gb
    elif ga is not gb:
        ga |= gb
        for i in gb:
            index_id_vers_grappe[i] = ga
        grappes.remove(gb)

print(f"{len(grappes)} grappes de doublons distinctes.")

a_supprimer = []
for grappe in grappes:
    lignes = [dict(conn.execute("SELECT * FROM entreprises WHERE id = ?", (i,)).fetchone()) for i in grappe]
    lignes.sort(key=score_completude, reverse=True)
    a_garder = lignes[0]
    for ligne in lignes[1:]:
        a_supprimer.append(ligne["id"])

print(f"{len(a_supprimer)} lignes à supprimer (doublons moins complets).")

2327 grappes de doublons distinctes.
2375 lignes à supprimer (doublons moins complets).


In [22]:
for grappe in grappes[:10]:
    lignes = [dict(conn.execute("SELECT * FROM entreprises WHERE id = ?", (i,)).fetchone()) for i in grappe]
    lignes.sort(key=score_completude, reverse=True)
    print(f"GARDÉ  : id={lignes[0]['id']:6} nom={lignes[0]['nom']:35} score={score_completude(lignes[0])}")
    for l in lignes[1:]:
        print(f"  supprimé : id={l['id']:6} nom={l['nom']:35} score={score_completude(l)}")
    print()

GARDÉ  : id=   113 nom=Hôpital Arassa                      score=2
  supprimé : id=  5011 nom=Hôpital Arassa                      score=1

GARDÉ  : id=   114 nom=polyclinique Souk El Tenine         score=3
  supprimé : id=  4803 nom=polyclinique Souk El Tenine         score=1

GARDÉ  : id=   115 nom=Hôpital orthopédique                score=2
  supprimé : id=  4804 nom=Hôpital orthopédique                score=1

GARDÉ  : id=   116 nom=عيادة متعددة الخدمات أيث شبانة      score=3
  supprimé : id=  5022 nom=عيادة متعددة الخدمات أيث شبانة      score=2

GARDÉ  : id=   117 nom=Oued achalal                        score=5
  supprimé : id=  4892 nom=Oued achalal                        score=2

GARDÉ  : id=   118 nom=Hôpital Souk El Tenine مستشفى سوق الاثنين score=3
  supprimé : id=  4805 nom=Hôpital Souk El Tenine مستشفى سوق الاثنين score=1

GARDÉ  : id=   119 nom=TAKLINIKT LEFLAY                    score=2
  supprimé : id=  4806 nom=TAKLINIKT LEFLAY                    score=1

GARDÉ  : id=   

In [25]:
import shutil
shutil.copy("../data/dasec_prospection.db", "../data/dasec_prospection_avant_dedup.db")
print("Sauvegarde créée : ../data/dasec_prospection_avant_dedup.db")

placeholders = ",".join("?" * len(a_supprimer))
conn.execute(f"DELETE FROM entreprises WHERE id IN ({placeholders})", a_supprimer)
conn.commit()

total_apres = conn.execute("SELECT COUNT(*) as n FROM entreprises").fetchone()["n"]
print(f"Suppression terminée. Total après nettoyage : {total_apres}")

Sauvegarde créée : ../data/dasec_prospection_avant_dedup.db
Suppression terminée. Total après nettoyage : 9994


In [26]:
import sqlite3
import unicodedata
import difflib

conn = sqlite3.connect("../data/dasec_prospection.db")
conn.row_factory = sqlite3.Row

def normaliser(texte):
    if not texte:
        return ""
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return texte.lower().strip()

cursor = conn.execute("""
    SELECT id, nom, secteur, sous_secteur, source_id,
           ROUND(latitude, 4) as lat_r, ROUND(longitude, 4) as lon_r
    FROM entreprises
    WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""")

par_position = {}
for row in cursor.fetchall():
    cle = (row["lat_r"], row["lon_r"], row["secteur"])
    par_position.setdefault(cle, []).append(dict(row))

vrais_doublons = []
for cle, groupe in par_position.items():
    if len(groupe) < 2:
        continue
    for i in range(len(groupe)):
        for j in range(i + 1, len(groupe)):
            n1, n2 = normaliser(groupe[i]["nom"]), normaliser(groupe[j]["nom"])
            similarite = difflib.SequenceMatcher(None, n1, n2).ratio()
            if similarite >= 0.85:  # noms quasi identiques uniquement
                vrais_doublons.append((groupe[i], groupe[j], round(similarite, 2)))

print(f"{len(vrais_doublons)} vrais doublons probables (nom similaire à 85%+ ET même position).")
for a, b, sim in vrais_doublons[:40]:
    print(f"  [{sim}] {a['id']} ({a['nom']}) <-> {b['id']} ({b['nom']})")

0 vrais doublons probables (nom similaire à 85%+ ET même position).


In [27]:
conn.execute("""
    UPDATE entreprises SET secteur = 'assurance'
    WHERE secteur = 'santé' AND sous_secteur = 'assurance privée'
""")
conn.commit()

In [28]:
cursor = conn.execute("""
    SELECT secteur, sous_secteur, COUNT(DISTINCT wilaya_name) as nb_wilayas,
           SUM(CASE WHEN wilaya_name IS NULL THEN 1 ELSE 0 END) as sans_wilaya,
           COUNT(*) as total
    FROM entreprises
    GROUP BY secteur, sous_secteur
    ORDER BY sans_wilaya DESC
""")
for row in cursor.fetchall():
    print(f"{row['secteur']:12} {row['sous_secteur'] or '—':30} {row['sans_wilaya']:5} sans wilaya / {row['total']:5} total")

santé        clinique privée                 1463 sans wilaya /  1793 total
étatique     mairie                           855 sans wilaya /  1125 total
santé        cabinet médical                  689 sans wilaya /  2168 total
santé        hôpital                          544 sans wilaya /  1142 total
assurance    assurance privée                 332 sans wilaya /   463 total
santé        cabinet dentaire                 248 sans wilaya /   479 total
industrie    entreprise industrielle          211 sans wilaya /   272 total
santé        pharmacie                        194 sans wilaya /  1752 total
juridique    cabinet d'avocat                 150 sans wilaya /   166 total
assurance    CNAS                              71 sans wilaya /    85 total
santé        opticien                          65 sans wilaya /   138 total
étatique     siège de wilaya                   30 sans wilaya /    79 total
assurance    CASNOS                            23 sans wilaya /    27 total
santé       

In [29]:
import osmium
print('area module:', dir(osmium.area))
print()
print('MultipolygonManager:', osmium.area.MultipolygonManager.__doc__)
print()

area module: ['AreaManager', 'AreaManagerBufferHandler', 'AreaManagerSecondPassHandler', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']



AttributeError: module 'osmium.area' has no attribute 'MultipolygonManager'